## Evaluation pipeline for the microlane experiment

In [16]:
# First consider all the variables
# The input image gets resized to a particular level
# Then create a pipeline to feed data into the model
# AFter this process is completed, then process the data
# Then after the processing is done find a way to take output from the model
# Then, convert the output to relevant format, and store it for future use
# Apply relevant computations

In [17]:
# Imports of the Core Packages
import json, sys, time, pytz
import os, gc, yaml, random, psutil
import numpy as np
from datetime import datetime
from PIL import Image
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

In [18]:
# Import custom libraries located at different folder location + configs
from microlane.utils.metrics import *
from microlane.datasets.tusimple import TuSimple
from microlane.datasets.raw import Raw
from microlane.models.lanenet2.model import LaneNet2
from microlane.models.ufld.model import UFLD
from microlane.schema.output import ModelPrediction
from microlane.schema.sample import Sample
from microlane.utils.load_image import load_image_from_sample, parse_prediction
from microlane.utils.experiment import ExperimentEvaluate

In [19]:
# First Load the Configuation file
with open("../configs/config.yaml", "r") as file:
    config = yaml.safe_load(file)

In [20]:
NUMBER = 10

### Pre Processing Part

In [21]:
# First initialise the dataset
# Then load the dataset
dataset = Raw(
    
    folder_path="/mnt/c/Users/suyog/Downloads/AI Images"
)

data = dataset.load_raw(number=10)

In [22]:
# So, basically now we will import the model
# model = LaneNet2() type and what we will do is, run 
# Run model.inference(formatted_dataset)
# Ensure that Docker Engine Is Running in background

# model = LaneNet2(
    
#     container_folder=config['models']['lanenet2']['container_folder'],
    
#     image_name=config['models']['lanenet2']['image_name']
    
# )

model = UFLD(
    
    container_folder=config['models']['ultra_fast_lane_detection']['container_folder'],
    
    image_name=config['models']['ultra_fast_lane_detection']['image_name']
)

Initializing container on port  8000
/home/suyog/desktop/projects/microlane/microlane/microlane/models/ufld/ufld
Image 'ufld_image:latest' already exists, skipping build.
Container already running: ffc63557c904


In [23]:
experiment = ExperimentEvaluate(
    
    experiment_name="BATCH pipeline testing with no augmentation with UFLD model on AI Generated data"
    
)

### Models and Datasets Loaded, Now Processing Part

In [24]:
# Print some basic information of our data

print(f"Total items: {len(data)}\n")

item = data[0]
print(f"Image Path   : {item.image_path}")
print(f"h_samples    : {item.h_samples}")
print(f"lanes        : {item.lanes}")

Total items: 10

Image Path   : /mnt/c/Users/suyog/Downloads/AI Images/frame_0000.png
h_samples    : [160.0, 170.0, 180.0, 190.0, 200.0, 210.0, 220.0, 230.0, 240.0, 250.0, 260.0, 270.0, 280.0, 290.0, 300.0, 310.0, 320.0, 330.0, 340.0, 350.0, 360.0, 370.0, 380.0, 390.0, 400.0, 410.0, 420.0, 430.0, 440.0, 450.0, 460.0, 470.0, 480.0, 490.0, 500.0, 510.0, 520.0, 530.0, 540.0, 550.0, 560.0, 570.0, 580.0, 590.0, 600.0, 610.0, 620.0, 630.0, 640.0, 650.0, 660.0, 670.0, 680.0, 690.0, 700.0, 710.0]
lanes        : []


In [25]:
blur_range = tuple(config['data']['augmentation']['blur'])
rotation_range = tuple(config['data']['augmentation']['blur'])
zoom_range = tuple(config['data']['augmentation']['zoom'])
lighting_range =tuple(config['data']['augmentation']['lighting'])

In [26]:
# numbers = [random.randint(0, NUMBER - 1) for _ in range(10)]
numbers = range(10)

In [27]:
print(numbers)

range(0, 10)


### Looping through all the testing examples

In [28]:
for index, testing_example in enumerate(tqdm(data)):
    
    
    ## Possible Data Augmentation that we could do here

    # item.blur       = round(random.uniform(blur_range[0],       blur_range[1] - 0.01), 2)
    # item.rotation   = round(random.uniform(rotation_range[0],   rotation_range[1] - 0.01), 2)
    # item.zoom       = round(random.uniform(zoom_range[0],       zoom_range[1] - 0.01), 2)
    # item.lighting = round(random.uniform(lighting_range[0], lighting_range[1] - 0.01), 2)
    # testing_example.motion_blur = 1

    loaded_image = load_image_from_sample(testing_example)
    
    response = model.predict(loaded_image)
    
    modeloutput = parse_prediction(response)
    
    experiment.store_prediction(modeloutput)
        
    if index in numbers:
        
        experiment.visualize_prediction(modeloutput)
    
    if (index % 100 == 0) and (index != 0):
        
        print(f"Routine Container Restart for Addressing Memory Leak [{int(index/100)}]")
        
        model.container_manager.restart_container()

  0%|          | 0/10 [00:00<?, ?it/s]